In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Introduction](introyt1_tutorial.html) \|\|
[Tensors](tensors_deeper_tutorial.html) \|\|
[Autograd](autogradyt_tutorial.html) \|\| [Building
Models](modelsyt_tutorial.html) \|\| **TensorBoard Support** \|\|
[Training Models](trainingyt.html) \|\| [Model
Understanding](captumyt.html)

PyTorch TensorBoard Support
===========================

Follow along with the video below or on
[youtube](https://www.youtube.com/watch?v=6CEld3hZgqc).



In [ ]:
# Run this cell to load the video
from IPython.display import display, HTML
html_code = """
<div style="margin-top:10px; margin-bottom:10px;">
  <iframe width="560" height="315" src="https://www.youtube.com/embed/6CEld3hZgqc" frameborder="0" allow="accelerometer; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>
</div>
"""
display(HTML(html_code))



Before You Start  
开始之前
----------------

To run this tutorial, you'll need to install PyTorch, TorchVision,
Matplotlib, and TensorBoard.  
要运行本教程，你需要安装 PyTorch、TorchVision、Matplotlib 和 TensorBoard。

With `pip`:  
使用 `pip` 安装：

``` {.sh}
pip install torch torchvision matplotlib tensorboard
```

Once the dependencies are installed, restart this notebook in the Python
environment where you installed them.  
依赖项安装完成后，请在安装了这些依赖项的 Python 环境中重启此 notebook。

Introduction  
简介
------------

In this notebook, we'll be training a variant of LeNet-5 against the
Fashion-MNIST dataset. Fashion-MNIST is a set of image tiles depicting
various garments, with ten class labels indicating the type of garment
depicted.  
在本 notebook 中，我们将使用 Fashion-MNIST 数据集来训练一个 LeNet-5 的变体模型。Fashion-MNIST 是一组描绘各类服装的图像集，包含十个类别标签，用于指示图像中所描绘的服装类型。


In [2]:
# PyTorch model and training necessities
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Image datasets and image manipulation
import torchvision
from torchvision.transforms import v2

# Image display
import matplotlib.pyplot as plt
import numpy as np

# PyTorch TensorBoard support
from torch.utils.tensorboard import SummaryWriter

Showing Images in TensorBoard  
在 TensorBoard 中显示图像
=============================

Let's start by adding sample images from our dataset to TensorBoard:  
首先，让我们将数据集中的示例图像添加到 TensorBoard 中：

In [ ]:
# Gather datasets and prepare them for consumption
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5,), (0.5,))])

# Store separate training and validations splits in ./data
training_set = torchvision.datasets.FashionMNIST('./data',
    download=True,
    train=True,
    transform=transform)
validation_set = torchvision.datasets.FashionMNIST('./data',
    download=True,
    train=False,
    transform=transform)

training_loader = torch.utils.data.DataLoader(training_set,
                                              batch_size=4,
                                              shuffle=True,
                                              num_workers=2)


validation_loader = torch.utils.data.DataLoader(validation_set,
                                                batch_size=4,
                                                shuffle=False,
                                                num_workers=2)

# Class labels
classes = ('T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
        'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle Boot')

# Helper function for inline image display
def matplotlib_imshow(img, one_channel=False):
    if one_channel:
        img = img.mean(dim=0)
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    if one_channel:
        plt.imshow(npimg, cmap="Greys")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))

# Extract a batch of 4 images
dataiter = iter(training_loader)
images, labels = next(dataiter)

# Create a grid from the images and show them
img_grid = torchvision.utils.make_grid(images)
matplotlib_imshow(img_grid, one_channel=True)

Above, we used TorchVision and Matplotlib to create a visual grid of a
minibatch of our input data. Below, we use the `add_image()` call on
`SummaryWriter` to log the image for consumption by TensorBoard, and we
also call `flush()` to make sure it's written to disk right away.  
在上面，我们使用了 TorchVision 和 Matplotlib 来创建一个输入数据小批次的可视化网格。在下面，我们使用 `SummaryWriter` 上的 `add_image()` 调用来记录图像，以供 TensorBoard 读取，并且我们还调用了 `flush()` 以确保数据立即写入磁盘。


In [ ]:
# Default log_dir argument is "runs" - but it's good to be specific
# torch.utils.tensorboard.SummaryWriter is imported above
writer = SummaryWriter('runs/fashion_mnist_experiment_1')

# Write image data to TensorBoard log dir
writer.add_image('Four Fashion-MNIST Images', img_grid)
writer.flush()

# To view, start TensorBoard on the command line with:
#   tensorboard --logdir=runs
# ...and open a browser tab to http://localhost:6006/

If you start TensorBoard at the command line and open it in a new
browser tab (usually at [localhost:6006](localhost:6006)), you should
see the image grid under the IMAGES tab.  
如果您在命令行启动 TensorBoard，并在新的浏览器标签页中打开它（通常访问地址为 localhost:6006），您应该能在 IMAGES（图像）选项卡下看到图像网格。

Graphing Scalars to Visualize Training  
绘制标量图表以可视化训练过程
======================================

TensorBoard is useful for tracking the progress and efficacy of your
training. Below, we'll run a training loop, track some metrics, and save
the data for TensorBoard's consumption.  
TensorBoard 对于跟踪训练进度和效果非常有用。下面，我们将运行一个训练循环，跟踪一些指标，并保存数据供 TensorBoard 读取。

Let's define a model to categorize our image tiles, and an optimizer and
loss function for training:  
让我们定义一个用于对图像进行分类的模型，以及用于训练的优化器和损失函数：

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 4 * 4)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    

net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

Now let's train a single epoch, and evaluate the training vs. validation
set losses every 1000 batches:  
现在，让我们训练一个 epoch（轮次），并每 1000 个批次评估一次训练集与验证集的损失：

In [ ]:
print(len(validation_loader))
for epoch in range(1):  # loop over the dataset multiple times
    running_loss = 0.0

    for i, data in enumerate(training_loader, 0):
        # basic training loop
        inputs, labels = data
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 1000 == 999:    # Every 1000 mini-batches...
            print(f'Batch {i + 1}')
            # Check against the validation set
            running_vloss = 0.0
            
            # In evaluation mode some model specific operations can be omitted eg. dropout layer
            net.eval() # Switching to evaluation mode, eg. turning off regularisation
            with torch.no_grad():
                for j, vdata in enumerate(validation_loader, 0):
                    vinputs, vlabels = vdata
                    voutputs = net(vinputs)
                    vloss = criterion(voutputs, vlabels)
                    running_vloss += vloss.item()
            net.train() # Switching back to training mode, eg. turning on regularisation
            
            avg_loss = running_loss / 1000
            avg_vloss = running_vloss / len(validation_loader)
            
            # Log the running loss averaged per batch
            writer.add_scalars('Training vs. Validation Loss',
                            { 'Training' : avg_loss, 'Validation' : avg_vloss },
                            epoch * len(training_loader) + i)

            running_loss = 0.0
print('Finished Training')

writer.flush()

Switch to your open TensorBoard and have a look at the SCALARS tab.  
切换到已打开的 TensorBoard，查看 SCALARS（标量）选项卡。

Visualizing Your Model  
可视化你的模型
======================

TensorBoard can also be used to examine the data flow within your model.  
TensorBoard 还可以用于检查模型内部的数据流。

To do this, call the `add_graph()` method with a model and sample input:  
为此，传入模型和样本输入，调用 `add_graph()` 方法：

In [ ]:
# Again, grab a single mini-batch of images
dataiter = iter(training_loader)
images, labels = next(dataiter)

# add_graph() will trace the sample input through your model,
# and render it as a graph.
writer.add_graph(net, images)
writer.flush()

When you switch over to TensorBoard, you should see a GRAPHS tab.
Double-click the "NET" node to see the layers and data flow within your
model.  
当你切换到 TensorBoard 时，应该能看到一个 GRAPHS（图）选项卡。
双击 "NET" 节点，即可查看模型内部的层结构和数据流向。

Visualizing Your Dataset with Embeddings  
使用嵌入（Embeddings）可视化数据集
========================================

The 28-by-28 image tiles we're using can be modeled as 784-dimensional
vectors (28 \* 28 = 784). It can be instructive to project this to a
lower-dimensional representation. The `add_embedding()` method will
project a set of data onto the three dimensions with highest variance,
and display them as an interactive 3D chart. The `add_embedding()`
method does this automatically by projecting to the three dimensions
with highest variance.  
我们使用的 28x28 的图像块可以被建模为 784 维向量（28 * 28 = 784）。将其投影到低维表示中进行观察是非常有启发性的。`add_embedding()` 方法会自动将一组数据投影到方差最大的三个维度上，并将其显示为一个交互式的 3D 图表。

Below, we'll take a sample of our data, and generate such an embedding:  
下面，我们将获取一个数据样本，并生成这样的嵌入投影：

In [ ]:
# Select a random subset of data and corresponding labels
def select_n_random(data, labels, n=100):
    assert len(data) == len(labels)

    perm = torch.randperm(len(data))
    return data[perm][:n], labels[perm][:n]

# Extract a random subset of data
images, labels = select_n_random(training_set.data, training_set.targets)

# get the class labels for each image
class_labels = [classes[label] for label in labels]

# log embeddings
features = images.view(-1, 28 * 28)
writer.add_embedding(features,
                    metadata=class_labels,
                    label_img=images.unsqueeze(1))
writer.flush()
writer.close()

Now if you switch to TensorBoard and select the PROJECTOR tab, you
should see a 3D representation of the projection. You can rotate and
zoom the model. Examine it at large and small scales, and see whether
you can spot patterns in the projected data and the clustering of
labels.  
现在，如果你切换到 TensorBoard 并选择 PROJECTOR（投影）选项卡，你应该能看到该投影的 3D 表示。你可以旋转和缩放模型。在不同的大尺度或小尺度下观察它，看看是否能在投影数据中发现规律，以及标签的聚类情况。

For better visibility, it's recommended to:  
为了获得更好的视觉效果，建议进行以下操作：

-   Select "label" from the "Color by" drop-down on the left.  
    在左侧的 "Color by"（按...着色）下拉菜单中选择 "label"（标签）。
-   Toggle the Night Mode icon along the top to place the light-colored
    images on a dark background.  
    点击顶部的 Night Mode（夜间模式）图标，将浅色图像放置在深色背景上。

Other Resources  
其他资源
===============

For more information, have a look at:  
如需了解更多信息，请查阅：

-   PyTorch documentation on
    [torch.utils.tensorboard.SummaryWriter](https://pytorch.org/docs/stable/tensorboard.html?highlight=summarywriter)
-   Tensorboard tutorial content in the [PyTorch.org
    Tutorials](https://pytorch.org/tutorials/)
-   For more information about TensorBoard, see the [TensorBoard
    documentation](https://www.tensorflow.org/tensorboard)
